In [1]:
import pandas as pd
from collections import defaultdict

In [2]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# show all columns
pd.set_option("display.max_columns", None)

In [3]:
%%javascript
// Disable auto-scrolling
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [4]:
%%R

require('tidyverse')
require('DescTools')

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: tidyverse
Loading required package: DescTools


In [5]:
# Load match data

import glob

# Step 1: Get all CSV files in the folder
csv_files = glob.glob("/Users/hazelgandhi/Desktop/tennis-regression/files/*.csv")

# Step 2: Read and concatenate them
df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

# Optional: Check the shape or preview
print(df.shape)
df.head()
df = df.sort_values('tourney_date')

five_set = ['Us Open','Roland Garros', 'Australian Open', 'Wimbledon']
df['five_set'] = df.tourney_name.isin(five_set)


# Initialize Elo ratings
elo_ratings = defaultdict(lambda: 1500)

# Store Elo snapshot *before* each match
elo_snapshots = []

K = 30

def win_prob(rating_i, rating_j):
    return 1 / (1 + 10 ** ((rating_j - rating_i) / 400))

for _, match in df.iterrows():
    winner = match['winner_name']
    loser = match['loser_name']

    rating_winner = elo_ratings[winner]
    rating_loser = elo_ratings[loser]

    # Record pre-match Elo ratings
    elo_snapshots.append({
        'date': match['tourney_date'],
        'surface': match['surface'],
        'tournament': match['tourney_name'],
        'winner': winner,
        'loser': loser,
        'five_set': match['five_set'],
        'winner_elo_before': rating_winner,
        'loser_elo_before': rating_loser
    })

    # Calculate expected outcomes
    expected_win = win_prob(rating_winner, rating_loser)
    expected_loss = 1 - expected_win

    # Update ratings
    elo_ratings[winner] += K * (1 - expected_win)
    elo_ratings[loser] += K * (0 - expected_loss)

# Create DataFrame
elo_df = pd.DataFrame(elo_snapshots)
elo_df

(8979, 49)


,date,surface,tournament,winner,loser,five_set,winner_elo_before,loser_elo_before
0,20220103,Hard,Adelaide 1,Juan Manuel Cerundolo,Alex Bolt,False,1500.000000,1500.000000
1,20220103,Hard,Adelaide 1,Marin Cilic,Thiago Monteiro,False,1500.000000,1500.000000
2,20220103,Hard,Adelaide 1,Laslo Djere,Corentin Moutet,False,1500.000000,1500.000000
3,20220103,Hard,Adelaide 1,Mikael Ymer,Soon Woo Kwon,False,1500.000000,1500.000000
4,20220103,Hard,Adelaide 1,Thanasi Kokkinakis,Frances Tiafoe,False,1500.000000,1500.000000
...,...,...,...,...,...,...,...,...
8974,20241218,Hard,Next Gen Finals,Luca Van Assche,Juncheng Shang,False,1420.401813,1582.974281
8975,20241218,Hard,Next Gen Finals,Nishesh Basavareddy,Juncheng Shang,False,1500.000000,1561.426509
8976,20241218,Hard,Next Gen Finals,Luca Van Assche,Nishesh Basavareddy,False,1441.949585,1517.624705
8977,20241218,Hard,Next Gen Finals,Learner Tien,Arthur Fils,False,1519.809918,1700.206746


In [6]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Randomly assign winner to be Player A or B
winner_is_A = np.random.randint(0, 2, size=len(elo_df)) == 1

# Create transformed DataFrame
elo_transformed = pd.DataFrame({
    'date': elo_df['date'],
    'tournament': elo_df['tournament'],
    'surface' : elo_df['surface'],
    'five_set': elo_df['five_set'],
    'player_A_name': np.where(winner_is_A, elo_df['winner'], elo_df['loser']),
    'player_B_name': np.where(winner_is_A, elo_df['loser'], elo_df['winner']),
    
    'player_A_elo_before': np.where(winner_is_A, elo_df['winner_elo_before'], elo_df['loser_elo_before']),
    'player_B_elo_before': np.where(winner_is_A, elo_df['loser_elo_before'], elo_df['winner_elo_before']),
    
    'player_A_win': winner_is_A.astype(int)
})
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win
0,20220103,Adelaide 1,Hard,False,Alex Bolt,Juan Manuel Cerundolo,1500.000000,1500.000000,0
1,20220103,Adelaide 1,Hard,False,Marin Cilic,Thiago Monteiro,1500.000000,1500.000000,1
2,20220103,Adelaide 1,Hard,False,Corentin Moutet,Laslo Djere,1500.000000,1500.000000,0
3,20220103,Adelaide 1,Hard,False,Soon Woo Kwon,Mikael Ymer,1500.000000,1500.000000,0
4,20220103,Adelaide 1,Hard,False,Frances Tiafoe,Thanasi Kokkinakis,1500.000000,1500.000000,0
...,...,...,...,...,...,...,...,...,...
8974,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Juncheng Shang,1420.401813,1582.974281,1
8975,20241218,Next Gen Finals,Hard,False,Juncheng Shang,Nishesh Basavareddy,1561.426509,1500.000000,0
8976,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Nishesh Basavareddy,1441.949585,1517.624705,1
8977,20241218,Next Gen Finals,Hard,False,Learner Tien,Arthur Fils,1519.809918,1700.206746,1


In [7]:
elo_transformed['elo_difference'] = elo_transformed['player_A_elo_before'] - elo_transformed ['player_B_elo_before']
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference
0,20220103,Adelaide 1,Hard,False,Alex Bolt,Juan Manuel Cerundolo,1500.000000,1500.000000,0,0.000000
1,20220103,Adelaide 1,Hard,False,Marin Cilic,Thiago Monteiro,1500.000000,1500.000000,1,0.000000
2,20220103,Adelaide 1,Hard,False,Corentin Moutet,Laslo Djere,1500.000000,1500.000000,0,0.000000
3,20220103,Adelaide 1,Hard,False,Soon Woo Kwon,Mikael Ymer,1500.000000,1500.000000,0,0.000000
4,20220103,Adelaide 1,Hard,False,Frances Tiafoe,Thanasi Kokkinakis,1500.000000,1500.000000,0,0.000000
...,...,...,...,...,...,...,...,...,...,...
8974,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Juncheng Shang,1420.401813,1582.974281,1,-162.572468
8975,20241218,Next Gen Finals,Hard,False,Juncheng Shang,Nishesh Basavareddy,1561.426509,1500.000000,0,61.426509
8976,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Nishesh Basavareddy,1441.949585,1517.624705,1,-75.675119
8977,20241218,Next Gen Finals,Hard,False,Learner Tien,Arthur Fils,1519.809918,1700.206746,1,-180.396828


### Trying initial logistic model

In [8]:
%%R -i elo_transformed

logistic <- glm(player_A_win ~ elo_difference + elo_difference:five_set, data=elo_transformed, family=binomial)
print(summary(logistic))
print(PseudoR2(logistic, which="McFadden"))


Call:
glm(formula = player_A_win ~ elo_difference + elo_difference:five_set, 
    family = binomial, data = elo_transformed)

Coefficients:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                 -0.0116483  0.0223175  -0.522    0.602    
elo_difference               0.0050482  0.0002208  22.864  < 2e-16 ***
elo_difference:five_setTRUE  0.0030059  0.0005455   5.510 3.58e-08 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 12447  on 8978  degrees of freedom
Residual deviance: 11426  on 8976  degrees of freedom
AIC: 11432

Number of Fisher Scoring iterations: 4

  McFadden 
0.08202639 


### Adding surface clusters

In [9]:
surface_stats = elo_transformed.groupby(['player_A_name', 'surface'])['player_A_win'].agg(['sum', 'count']).reset_index()
surface_stats.columns = ['player', 'surface', 'wins', 'matches']
surface_stats['win_pct'] = surface_stats['wins'] / surface_stats['matches']

surface_stats

,player,surface,wins,matches,win_pct
0,Abedallah Shelbayh,Clay,0,1,0.000000
1,Abedallah Shelbayh,Grass,0,2,0.000000
2,Abedallah Shelbayh,Hard,2,5,0.400000
3,Adam Moundir,Clay,0,1,0.000000
4,Adam Neff,Hard,0,1,0.000000
...,...,...,...,...,...
997,Zizou Bergs,Clay,1,6,0.166667
998,Zizou Bergs,Grass,0,2,0.000000
999,Zizou Bergs,Hard,5,15,0.333333
1000,Zsombor Piros,Clay,2,3,0.666667


### Pivoting the table to wide format

In [10]:
surface_wide = surface_stats.pivot(index='player', columns='surface', values='win_pct').fillna(0)
surface_wide.columns = [f"{col}_win_pct" for col in surface_wide.columns]
surface_wide.reset_index(inplace=True)
surface_wide

,player,Clay_win_pct,Grass_win_pct,Hard_win_pct
0,Abedallah Shelbayh,0.000000,0.000000,0.400000
1,Adam Moundir,0.000000,0.000000,0.000000
2,Adam Neff,0.000000,0.000000,0.000000
3,Adam Walton,0.000000,0.500000,0.166667
4,Adria Soriano Barrera,1.000000,0.000000,0.000000
...,...,...,...,...
526,Zachary Svajda,0.000000,0.000000,0.272727
527,Zdenek Kolar,1.000000,0.000000,0.000000
528,Zhizhen Zhang,0.470588,0.666667,0.379310
529,Zizou Bergs,0.166667,0.000000,0.333333


In [11]:
## Adding clusters
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = surface_wide[['Clay_win_pct', 'Grass_win_pct', 'Hard_win_pct']]
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

surface_wide['surface_cluster'] = clusters
surface_wide

,player,Clay_win_pct,Grass_win_pct,Hard_win_pct,surface_cluster
0,Abedallah Shelbayh,0.000000,0.000000,0.400000,3
1,Adam Moundir,0.000000,0.000000,0.000000,0
2,Adam Neff,0.000000,0.000000,0.000000,0
3,Adam Walton,0.000000,0.500000,0.166667,1
4,Adria Soriano Barrera,1.000000,0.000000,0.000000,2
...,...,...,...,...,...
526,Zachary Svajda,0.000000,0.000000,0.272727,0
527,Zdenek Kolar,1.000000,0.000000,0.000000,2
528,Zhizhen Zhang,0.470588,0.666667,0.379310,1
529,Zizou Bergs,0.166667,0.000000,0.333333,0


In [12]:
player_clusters = surface_wide[['player', 'surface_cluster']]


In [13]:
# For player A
elo_transformed = elo_transformed.merge(player_clusters, left_on='player_A_name', right_on='player', how='left')
elo_transformed.rename(columns={'surface_cluster': 'surface_cluster_A'}, inplace=True)
elo_transformed.drop(columns='player', inplace=True)

# For player B
player_clusters.columns = ['player', 'surface_cluster_B']
elo_transformed = elo_transformed.merge(player_clusters, left_on='player_B_name', right_on='player', how='left')
elo_transformed.drop(columns='player', inplace=True)
elo_transformed


,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference,surface_cluster_A,surface_cluster_B
0,20220103,Adelaide 1,Hard,False,Alex Bolt,Juan Manuel Cerundolo,1500.000000,1500.000000,0,0.000000,1.0,2.0
1,20220103,Adelaide 1,Hard,False,Marin Cilic,Thiago Monteiro,1500.000000,1500.000000,1,0.000000,1.0,2.0
2,20220103,Adelaide 1,Hard,False,Corentin Moutet,Laslo Djere,1500.000000,1500.000000,0,0.000000,1.0,1.0
3,20220103,Adelaide 1,Hard,False,Soon Woo Kwon,Mikael Ymer,1500.000000,1500.000000,0,0.000000,2.0,1.0
4,20220103,Adelaide 1,Hard,False,Frances Tiafoe,Thanasi Kokkinakis,1500.000000,1500.000000,0,0.000000,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8974,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Juncheng Shang,1420.401813,1582.974281,1,-162.572468,1.0,3.0
8975,20241218,Next Gen Finals,Hard,False,Juncheng Shang,Nishesh Basavareddy,1561.426509,1500.000000,0,61.426509,3.0,NaN
8976,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Nishesh Basavareddy,1441.949585,1517.624705,1,-75.675119,1.0,NaN
8977,20241218,Next Gen Finals,Hard,False,Learner Tien,Arthur Fils,1519.809918,1700.206746,1,-180.396828,3.0,1.0


### Now the logistic model again

In [14]:
elo_transformed = elo_transformed.dropna()

In [15]:
elo_transformed

,date,tournament,surface,five_set,player_A_name,player_B_name,player_A_elo_before,player_B_elo_before,player_A_win,elo_difference,surface_cluster_A,surface_cluster_B
0,20220103,Adelaide 1,Hard,False,Alex Bolt,Juan Manuel Cerundolo,1500.000000,1500.000000,0,0.000000,1.0,2.0
1,20220103,Adelaide 1,Hard,False,Marin Cilic,Thiago Monteiro,1500.000000,1500.000000,1,0.000000,1.0,2.0
2,20220103,Adelaide 1,Hard,False,Corentin Moutet,Laslo Djere,1500.000000,1500.000000,0,0.000000,1.0,1.0
3,20220103,Adelaide 1,Hard,False,Soon Woo Kwon,Mikael Ymer,1500.000000,1500.000000,0,0.000000,2.0,1.0
4,20220103,Adelaide 1,Hard,False,Frances Tiafoe,Thanasi Kokkinakis,1500.000000,1500.000000,0,0.000000,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8971,20241218,Next Gen Finals,Hard,False,Jakub Mensik,Joao Fonseca,1636.355726,1574.533754,0,61.821973,1.0,3.0
8972,20241218,Next Gen Finals,Hard,False,Alex Michelsen,Juncheng Shang,1591.043124,1598.286979,1,-7.243855,1.0,3.0
8973,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Alex Michelsen,1428.324437,1606.355821,0,-178.031384,1.0,1.0
8974,20241218,Next Gen Finals,Hard,False,Luca Van Assche,Juncheng Shang,1420.401813,1582.974281,1,-162.572468,1.0,3.0


In [16]:
%%R -i elo_transformed

logistic_new <- glm(player_A_win ~ elo_difference + elo_difference:five_set + surface:factor(surface_cluster_A) + surface:factor(surface_cluster_B), data=elo_transformed, family=binomial)
print(summary(logistic_new))
print(PseudoR2(logistic_new, which="McFadden"))


Call:
glm(formula = player_A_win ~ elo_difference + elo_difference:five_set + 
    surface:factor(surface_cluster_A) + surface:factor(surface_cluster_B), 
    family = binomial, data = elo_transformed)

Coefficients: (1 not defined because of singularities)
                                          Estimate Std. Error z value Pr(>|z|)
(Intercept)                              1.0556772  0.1591431   6.634 3.28e-11
elo_difference                           0.0043429  0.0002328  18.655  < 2e-16
elo_difference:five_setTRUE              0.0029320  0.0005513   5.318 1.05e-07
surfaceClay:factor(surface_cluster_A)0  -3.1005039  0.3544877  -8.746  < 2e-16
surfaceGrass:factor(surface_cluster_A)0 -2.2303576  0.4582848  -4.867 1.13e-06
surfaceHard:factor(surface_cluster_A)0  -1.9981602  0.1836202 -10.882  < 2e-16
surfaceClay:factor(surface_cluster_A)1  -0.6562900  0.2358110  -2.783 0.005384
surfaceGrass:factor(surface_cluster_A)1 -0.2930030  0.3529271  -0.830 0.406421
surfaceHard:factor(surface_clu

### Checking prediction

In [17]:
%%R -i elo_transformed

df <- elo_transformed %>% mutate(
    predict_proba_R = predict(logistic_new, type="response"),
    predict_R = ifelse(predict_proba_R > .5, 1,0)
) %>% arrange(elo_difference)

df %>% head()

         date           tournament surface five_set       player_A_name
7419 20240527        Roland Garros    Clay     TRUE     Richard Gasquet
6703 20240304 Indian Wells Masters    Hard    FALSE          Luca Nardi
7384 20240527        Roland Garros    Clay     TRUE Christopher Eubanks
6050 20240115      Australian Open    Hard     TRUE        Dino Prizmic
6734 20240304 Indian Wells Masters    Hard    FALSE    Aleksandar Vukic
4705 20230703            Wimbledon   Grass     TRUE       Jeremy Chardy
      player_B_name player_A_elo_before player_B_elo_before player_A_win
7419  Jannik Sinner            1428.232            1990.804            0
6703 Novak Djokovic            1426.978            1988.387            1
7384  Jannik Sinner            1460.359            1994.272            0
6050 Novak Djokovic            1469.346            1989.502            0
6734 Novak Djokovic            1456.782            1959.527            0
4705 Carlos Alcaraz            1471.108            1973.75

In [18]:
%%R

library(dplyr)

df_filtered <- df %>%
  filter(player_A_win != predict_R) %>%
  arrange(desc(predict_proba_R))

head(df_filtered)

         date      tournament surface five_set         player_A_name
6093 20240115 Australian Open    Hard     TRUE       Grigor Dimitrov
4382 20230529   Roland Garros    Clay     TRUE         Jannik Sinner
2231 20220829         Us Open    Hard     TRUE          Taylor Fritz
7630 20240701       Wimbledon   Grass     TRUE       Karen Khachanov
6161 20240115 Australian Open    Hard     TRUE           Holger Rune
3017 20230102      Adelaide 1    Hard    FALSE Felix Auger Aliassime
       player_B_name player_A_elo_before player_B_elo_before player_A_win
6093     Nuno Borges            1845.968            1468.654            0
4382 Daniel Altmaier            1778.716            1404.601            0
2231    Brandon Holt            1685.448            1484.182            0
7630   Quentin Halys            1688.870            1480.473            0
6161   Arthur Cazaux            1729.388            1499.967            0
3017  Alexei Popyrin            1777.530            1399.370            0

In [19]:
%%R 

# Create confusion matrix
conf_mat <- table(df$predict_R, df$player_A_win)
print(conf_mat) 

# Extract TP, FP, FN
TP <- conf_mat[2,2]
FP <- conf_mat[2,1]
FN <- conf_mat[1,2]
# Calculate Precision and Recall
precision <- TP / (TP + FP)
recall <- TP / (TP + FN)
# Print
cat("Precision:", precision, "\n")
cat("Recall:", recall, "\n")

   
       0    1
  0 2819 1334
  1 1629 2989
Precision: 0.6472499 
Recall: 0.691418 


In [20]:
%%R
write.csv(df, "mens-analysis1-df.csv", row.names = FALSE)